# KAN-DI in Colab: descriptor importance for a table of experiments

This notebook runs `analyze.py` from [nicechoi99/kan-di](https://github.com/nicechoi99/kan-di) without a local install.
A Kolmogorov–Arnold Network (KAN) is fitted to your experiments, pruned to a closed-form expression, and each
descriptor is scored by its average gradient energy (AGE) under that expression.

Run the cells from top to bottom. The default Colab CPU runtime is enough; a GPU is not needed.

## 1. Get the code

In [ ]:
!git clone --depth 1 https://github.com/nicechoi99/kan-di.git
# To try the development branch instead:
# !git clone --depth 1 --branch readme-draft https://github.com/nicechoi99/kan-di.git
%cd kan-di

## 2. Install the dependencies

The CPU build of PyTorch is enough here. The other pins come from `requirements.txt`.
If Colab asks you to restart the session after the install, restart it and run the `%cd kan-di` line again.

In [ ]:
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r requirements.txt

## 3. Run on a benchmark pool from the paper

`dat/` holds the nine candidate pools of the paper's benchmarks (sources and licenses in `dat/README.md`).
AutoAM is the smallest: 100 extrusion prints, four printer settings (`Prime Delay`, `Print Speed`,
`X Offset Correction`, `Y Offset Correction`) and the print-quality `Score`. The paper finds that the two
offset corrections lead on this pool.
One KAN fit (`--fits 1`) keeps this to a few minutes; the campaign replay (`--simulate`) is left out because
it takes several minutes per seed.

In [ ]:
!python analyze.py dat/small_feature/AutoAM.csv --target Score --log-inputs on --fits 1 --out results_example

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv('results_example/descriptor_importance.csv'))
print(open('results_example/report.txt').read())
display(Image('results_example/summary.png'))

## 4. Analyze your own table

Upload a CSV or Excel file with one row per experiment, numeric descriptor columns and one response column.
Then set `TARGET` to the response column. Every other column is used as a descriptor unless you list
them in `FEATURES`. Rows with missing or non-numeric values are dropped and counted in the output.

In [ ]:
from google.colab import files

uploaded = files.upload()          # choose your file in the dialog
TABLE = next(iter(uploaded))       # name of the uploaded file
print(TABLE)
print(pd.read_csv(TABLE).head() if not TABLE.lower().endswith(('.xlsx', '.xls')) else pd.read_excel(TABLE).head())

In [ ]:
TARGET = 'Score'      # response column
FEATURES = ''         # e.g. 'T,P,ratio'; empty = all other columns
MINIMIZE = False      # True if a lower response is better
FITS = 1              # KAN fits averaged; more fits take longer

args = f'--target "{TARGET}" --fits {FITS} --out results_mine'
if FEATURES:
    args += f' --features "{FEATURES}"'
if MINIMIZE:
    args += ' --minimize'
!python analyze.py "{TABLE}" {args}

In [ ]:
display(pd.read_csv('results_mine/descriptor_importance.csv'))
print(open('results_mine/report.txt').read())
display(Image('results_mine/summary.png'))

# files.download('results_mine/descriptor_importance.csv')   # uncomment to save the table

The report warns when the closed-form expression explains less than half of the variance of the response;
a ranking from an expression that does not fit means little. Adding `--simulate --seeds 5` replays
campaigns of KAN-DI and a zero-mean GP on your table. The replay takes several minutes per seed, and its
iteration counts describe the rows you supplied, not experiments you have not run.

## 5. Optional: the quickstart

`benchmark/quickstart.py` runs the KAN-DI loop on a synthetic Hartmann-6 pool padded with four inputs that do not
enter the response. It takes a few minutes per seed. It checks that the implementation runs; it is not a
reproduction of the paper's results (see the README).

In [ ]:
# !cd benchmark && python quickstart.py